# Healthcare analysis walkthrough

## Goal
Inspect the frozen real-data cohort, verify its grain and monetary totals, and understand the SQL peer comparison. The Power BI browser report is a later, user-owned step.


## Setup
This companion uses Python standard-library modules only. Run from the project root or the notebooks folder. The prepared SQLite database and frozen raw CSV are included. To rebuild those files, use the documented pandas preparation script.


In [1]:
from pathlib import Path
import sqlite3, json, csv
from decimal import Decimal
root = Path.cwd()
if not (root / 'analysis').exists():
    root = root.parent
assert (root / 'analysis/healthcare.sqlite').exists()
con = sqlite3.connect(root / 'analysis/healthcare.sqlite')
con.row_factory = sqlite3.Row
print('Connected to the frozen project database.')


Connected to the frozen project database.


## Steps
### 1. Source and cohort
One row is one discharge, not one unique patient. Source: https://health.data.ny.gov/d/sf4k-39ay. The exact query and SHA-256 checksum are saved with the extract.


In [2]:
manifest = json.loads((root / 'source/download_manifest.json').read_text())
print('Dataset:', manifest['dataset_id'])
print('Filter:', manifest['where'])
print('Rows:', manifest['row_count'])
print('Retrieved:', manifest['retrieved_at_utc'])


Dataset: sf4k-39ay
Filter: health_service_area='New York City' AND apr_drg_code IN ('194','139','720') AND age_group != '0-17'
Rows: 89984
Retrieved: 2026-09-04T09:58:07.042058+00:00


### 2. Check the grain and censored stays
Capped 120+ day stays remain in counts and costs. They have no exact LOS value. Duplicate-looking visible records are retained if source IDs differ.


In [3]:
row = con.execute('SELECT COUNT(*) n, COUNT(DISTINCT SourceRowId) ids, COUNT(ExactLOS) exact_n, SUM(LOSCensored) capped_n FROM discharge_model').fetchone()
print(dict(row))
assert row['n'] == row['ids'] == manifest['row_count']
assert row['exact_n'] + row['capped_n'] == row['n']


{'n': 89984, 'ids': 89984, 'exact_n': 89922, 'capped_n': 62}


### 3. Compare clinical groups
The following query comes from the saved analysis SQL. Costs are estimated nominal USD. Mean LOS excludes censored values.


In [4]:
statements=[]
buffer=''
for line in (root / 'sql/02_analysis.sql').read_text().splitlines(True):
    buffer += line
    if sqlite3.complete_statement(buffer):
        statements.append(buffer)
        buffer=''
for row in con.execute(statements[0]):
    print(dict(row))


{'ConditionDescription': 'SEPTICEMIA AND DISSEMINATED INFECTIONS', 'Discharges': 55248, 'AvgExactLOS': 8.79, 'EstimatedCost': 2476236690.13, 'AvgEstimatedCost': 44820.39}
{'ConditionDescription': 'HEART FAILURE', 'Discharges': 25040, 'AvgExactLOS': 6.41, 'EstimatedCost': 759632578.03, 'AvgEstimatedCost': 30336.76}
{'ConditionDescription': 'OTHER PNEUMONIA', 'Discharges': 9696, 'AvgExactLOS': 5.3, 'EstimatedCost': 241787641.01, 'AvgEstimatedCost': 24936.84}


### 4. Examine the peer baseline
Peers share APR DRG, severity and age band, and exclude the focal hospital. A minimum of 30 peer records across three other hospitals is an analyst-chosen support rule. It is not a clinical standard.


In [5]:
row = con.execute('SELECT SUM(BenchmarkEligible) matched, COUNT(*) total, SUM(CASE WHEN BenchmarkEligible=1 THEN ExactLOS END) observed, SUM(ExpectedLOS) expected FROM discharge_model').fetchone()
print(dict(row))
print('Coverage:', round(row['matched']/row['total']*100,4), '%')
print('LOS index:', round(row['observed']/row['expected'],6))
assert row['matched'] <= row['total']


{'matched': 89877, 'total': 89984, 'observed': 696468, 'expected': 696827.2086807302}
Coverage: 99.8811 %
LOS index: 0.999485


## Checks
### 5. Reconcile estimated costs independently in cents
The comparison below starts from the unchanged source CSV and the SQL fact, using separate calculation paths.


In [6]:
with (root / 'data/raw/nyc_adult_selected_discharges_2024.csv').open() as f:
    raw_cents = sum(int(Decimal(r['total_costs'])*100) for r in csv.DictReader(f))
sql_cents = con.execute('SELECT SUM(CostCents) FROM discharge_model').fetchone()[0]
assert raw_cents == sql_cents
print('Reconciled estimated cost:', Decimal(raw_cents)/100)
validation = json.loads((root / 'qa/analysis_validation.json').read_text())
print('Independently checked peer strata:', validation['all_peer_strata_reconciled'])


Reconciled estimated cost: 3477656909.17
Independently checked peer strata: 2069


### 6. Check scenario arithmetic
A scenario is an assumed percentage of matched observed days, not a forecast or actual improvement.


In [7]:
days = con.execute('SELECT SUM(ExactLOS) FROM discharge_model WHERE BenchmarkEligible=1').fetchone()[0]
for rate in [0,.05,.10]:
    print(f'{rate:.0%}: {days*rate:,.1f} hypothetical bed-days')
assert days*.10 == 2*(days*.05)
con.close()


0%: 0.0 hypothetical bed-days
5%: 34,823.4 hypothetical bed-days
10%: 69,646.8 hypothetical bed-days


## Next Steps
Build the report using docs/POWER_BI_WEB_GUIDE.md. Check it against qa/powerbi_expected_results.csv. The selected cohort has 89,984 discharges, a highly skewed cost distribution, and substantial LOS variation by severity. Peer adjustment improves comparability but does not establish causal efficiency. Read the findings and methodology before presenting the results.
